In [4]:
# ====== CẤU HÌNH TOÀN CỤC ======
from pathlib import Path
import time, os

# Đường dẫn dữ liệu (đúng cấu trúc bạn yêu cầu)
DATA_DIR  = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/Images")
TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR   = DATA_DIR / "Validate"
TEST_DIR  = DATA_DIR / "Test"

# Tham số train
SEED           = 1337
NUM_CLASSES    = None            # sẽ tự suy ra từ dataset
IMG_SIZE       = 224
EPOCHS         = 100
BATCH_SIZE     = 64
LR             = 3e-4
WEIGHT_DECAY   = 1e-4
DROPOUT        = 0.4
PATIENCE       = 15               # ngừng train nếu val acc không cải thiện 5 epoch
PRETRAINED     = True


from multiprocessing import cpu_count
#NUM_WORKERS = max(2, cpu_count() // 2)
NUM_WORKERS    = 8 


# Tên checkpoint (theo yêu cầu, giữ nguyên spelling "effcientnet")
CKPT_BEST_NAME = "mtl_effcientnet_b0_best.pt"
CKPT_LAST_NAME = "mtl_effcientnet_b0_last.pt"

# Thư mục lưu log/charts
'''
RUN_DIR  = Path("runs") / time.strftime("%Y%m%d-%H%M%S")
CKPT_DIR = RUN_DIR / "checkpoints"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
'''


# ====== TÊN MODEL & CẤU TRÚC THƯ MỤC ======
MODEL_NAME = "mtl_efficientnet_b0"          # đổi theo model khác nếu cần
STAMP = time.strftime("%Y%m%d-%H%M%S")

RUN_DIR    = Path("runs") / f"{MODEL_NAME}_{STAMP}"
CKPT_DIR   = RUN_DIR / "checkpoints"
IMAGES_DIR = RUN_DIR / "images"
for p in [RUN_DIR, CKPT_DIR, IMAGES_DIR]:
    p.mkdir(parents=True, exist_ok=True)
# CSV ở RUN_DIR như cũ
CSV_PATH = RUN_DIR / "metrics.csv"



# Chuẩn hóa ImageNet + ép RGB để dẹp cảnh báo PIL (palette/transparency)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Cấu hình annotate confusion matrix
# Chỉ annotate các ô SAI có tỷ lệ sai (theo hàng) >= ngưỡng này
ANNOTATE_MIN_ERROR_PCT = 5.0  # ví dụ 10 nghĩa là tô/ghi các ô sai từ 10% trở lên


# ====== IMPORT & SETUP ======
import sys, random, json, math, gc, csv
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import itertools

# Táo bạo nhưng an toàn: seed cho tái lập + bật cudnn benchmark
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Tự chèn đường dẫn 'model/' để import file model của bạn (notebook đặt trong Jupyter/, model đặt trong ../model)
from pathlib import Path
ROOT_DIR  = Path.cwd() if (Path.cwd() / "model").exists() else Path.cwd().parent
MODEL_DIR = ROOT_DIR / "model"
sys.path.append(str(MODEL_DIR))

print("Model dir:", MODEL_DIR.resolve())


Device: cuda
Model dir: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/model


In [5]:
# Import đúng file bạn đã cung cấp
from models.mtl_efficientnet_b0 import MTLEfficientNetB0

# Khởi tạo model (dropout tùy chỉnh); để compile + channels_last + AMP cho nhanh
def build_model(num_classes: int,
                dropout: float = 0.4,
                pretrained: bool = True):
    model = MTLEfficientNetB0(
        num_classes=num_classes,
        pretrained=pretrained,
        freeze_backbone=False,
        dropout_rate=dropout
    )
    # channels_last tăng thông lượng trên Ampere/Turing
    model = model.to(device).to(memory_format=torch.channels_last)
    # PyTorch 2.x: compile để tối ưu kernel
    try:
        model = torch.compile(model, mode="reduce-overhead", fullgraph=False)
        print("✅ torch.compile enabled")
    except Exception as e:
        print("⚠️ torch.compile skipped:", e)
    return model

# Hàm tạo optimizer & scheduler (AdamW + CosineAnnealing)
def build_optim_sched(model, lr=3e-4, weight_decay=1e-4, epochs=EPOCHS):
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    return optimizer, scheduler


In [6]:
# Ép tất cả ảnh về RGB trước khi ToTensor để dẹp warning PIL (palette/transparency)
to_rgb = transforms.Lambda(lambda im: im.convert("RGB"))

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2,0.1,0.05),
    to_rgb,
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

'''
Ý nghĩa:
RandomResizedCrop: cắt ngẫu nhiên vùng 85–100% diện tích ảnh → mô hình học texture vùng trong (bánh, topping...).
ColorJitter: thay đổi sáng – tương phản – hue nhẹ → học được các điều kiện ánh sáng khác nhau (bếp, quán ăn...).
GaussianBlur: giúp bền hơn khi ảnh out-focus hoặc nhiễu.
RandomAffine: dịch chuyển và shear nhỏ giúp không phụ thuộc góc chụp.
'''
from torchvision import transforms

# === Data Augmentation kiểu "local-texture" cho training ===
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),

    # 🔸 Các phép biến đổi ánh sáng & texture cục bộ
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=5),

    # 🔸 Một số noise / blur nhẹ để giúp mô hình bền vững
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.0))], p=0.2),

    # 🔸 Chuẩn hóa cuối cùng
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    to_rgb,
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Đọc dataset theo cấu trúc thư mục
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=eval_tfms)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=eval_tfms)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print("Số lớp:", NUM_CLASSES)
print("Ví dụ lớp:", CLASS_NAMES[:30])




from torch.utils.data import WeightedRandomSampler
import numpy as np

# Đếm số mẫu theo class trong train_ds
class_counts = np.bincount(train_ds.targets)
weights = 1.0 / class_counts
sample_weights = [weights[t] for t in train_ds.targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_ds), replacement=True)
# → Các lớp ít sẽ được chọn ngẫu nhiên nhiều lần hơn → hiệu quả như oversampling. Mục tiêu là tăng cường những lớp ít ảnh hơn để tránh bias. nhân đôi lớp nhỏ tới ~1,000 ảnh
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=True)

# DataLoader: dùng nhiều worker, pin_memory, prefetch để feed GPU
#train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
#                          num_workers=NUM_WORKERS, pin_memory=True,
#                          persistent_workers=True, prefetch_factor=4)

val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)

test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)

# SAVE runs_meta
os.makedirs("runs_meta", exist_ok=True)
with open("runs_meta/class_names.json", "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

with open("runs_meta/mean_std.json", "w") as f:
    json.dump({"mean": IMAGENET_MEAN, "std": IMAGENET_STD}, f, indent=2)

with open("runs_meta/img_size.json", "w") as f:
    json.dump({"img_size": IMG_SIZE}, f, indent=2)

print("Saved runs_meta/")


Số lớp: 33
Ví dụ lớp: ['Banh beo', 'Banh bot loc', 'Banh can', 'Banh canh', 'Banh chung', 'Banh cuon', 'Banh duc', 'Banh gio', 'Banh khot', 'Banh mi', 'Banh pia', 'Banh tet', 'Banh trang nuong', 'Banh xeo', 'Bun bo Hue', 'Bun dau mam tom', 'Bun mam', 'Bun rieu', 'Bun thit nuong', 'Ca kho to', 'Canh chua', 'Cao lau', 'Chao long', 'Com tam', 'Goi cuon', 'Hu tieu', 'Mi quang', 'Nem chua', 'Pho', 'Xoi xeo']
Saved runs_meta/


In [7]:
# Xây model & tối ưu
model = build_model(NUM_CLASSES, dropout=DROPOUT, pretrained=PRETRAINED)
criterion = nn.CrossEntropyLoss()
optimizer, scheduler = build_optim_sched(model, lr=LR, weight_decay=WEIGHT_DECAY, epochs=EPOCHS)

# Scaler cho mixed-precision (AMP)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))

# Hàm tính accuracy nhanh
def accuracy_from_logits(logits, targets):
    preds = torch.argmax(logits, dim=1)
    return (preds == targets).float().mean().item()

# 1 epoch Train/Val (dùng torch.amp.autocast('cuda') để tránh cảnh báo deprecated)
def run_one_epoch(dataloader, is_train: bool = True):
    model.train(is_train)
    total_loss, total_acc, total_samples = 0.0, 0.0, 0

    for imgs, labels in dataloader:
        imgs   = imgs.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast('cuda' if device.type=='cuda' else 'cpu'):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = labels.size(0)
        total_samples += bs
        total_loss    += loss.item() * bs
        total_acc     += accuracy_from_logits(logits, labels) * bs

    return total_loss/total_samples, total_acc/total_samples


✅ torch.compile enabled


/tmp/ipykernel_15701/115215440.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))


In [8]:
# === PROGRESS BAR hiển thị %/ETA rõ trong VSCode ===
import os, sys, math, time, csv, json
os.environ["TQDM_NOTEBOOK"] = "1"  # ép dùng renderer notebook
from tqdm.notebook import tqdm      # notebook renderer đẹp hơn autonotebook

def run_one_epoch_pb(dataloader, is_train: bool, epoch: int, num_epochs: int):
    """
    Train/Valid 1 epoch với progress bar to, có %/ETA.
    """
    model.train(is_train)
    total_loss, total_acc, total_samples = 0.0, 0.0, 0

    pbar = tqdm(total=len(dataloader), leave=False, dynamic_ncols=True,
                desc=f"[{'Train' if is_train else 'Valid'}] Epoch {epoch}/{num_epochs}",
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")
    for step, (imgs, labels) in enumerate(dataloader):
        imgs   = imgs.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast('cuda' if device.type=='cuda' else 'cpu'):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = labels.size(0)
        total_samples += bs
        total_loss    += loss.item() * bs
        total_acc     += (logits.argmax(1) == labels).float().sum().item()

        avg_loss = total_loss / max(1, total_samples)
        avg_acc  = total_acc  / max(1, total_samples)
        pbar.set_postfix_str(f"loss={avg_loss:.4f} acc={avg_acc:.4f}")
        pbar.update(1)

    pbar.close()
    return total_loss/total_samples, total_acc/total_samples


# === TRAIN LOOP với progress bar ngoài theo epoch ===
from tqdm.notebook import tqdm as tqdm_outer


# 1) Tạo file CSV (nếu chưa có) để ghi metrics mỗi epoch
CSV_PATH = RUN_DIR / "metrics.csv"
if not CSV_PATH.exists():
    with open(CSV_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch","train_loss","val_loss","train_acc","val_acc","lr","time_s"])
print("CSV log  →", CSV_PATH)


best_val_acc, no_improve = -1.0, 0
history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}

outer = tqdm_outer(range(1, EPOCHS+1), total=EPOCHS, dynamic_ncols=True, desc="Epochs")
for epoch in outer:
    t0 = time.time()

    train_loss, train_acc = run_one_epoch_pb(train_loader, True,  epoch, EPOCHS)
    with torch.no_grad():
        val_loss,   val_acc = run_one_epoch_pb(val_loader,   False, epoch, EPOCHS)

    scheduler.step()
    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)

    outer.set_postfix_str(f"train_acc={train_acc:.4f} val_acc={val_acc:.4f} lr={lr_now:.2e} t={elapsed:.1f}s")

    with open(CSV_PATH, "a", newline="") as f:
        csv.writer(f).writerow([epoch, f"{train_loss:.6f}", f"{val_loss:.6f}",
                                f"{train_acc:.6f}", f"{val_acc:.6f}",
                                f"{lr_now:.8f}", f"{elapsed:.2f}"])

    # save LAST
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "class_names": CLASS_NAMES
    }, CKPT_DIR / CKPT_LAST_NAME)

    # save BEST theo val_acc
    if val_acc > best_val_acc:
        best_val_acc, no_improve = val_acc, 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "class_names": CLASS_NAMES
        }, CKPT_DIR / CKPT_BEST_NAME)
        outer.write(f"✅ New BEST (val_acc={best_val_acc:.4f}) → {CKPT_BEST_NAME}")
    else:
        no_improve += 1
        outer.write(f"(no improve {no_improve}/{PATIENCE})")

    if no_improve >= PATIENCE:
        outer.write("⏹ Early stopping (no val acc improvement)")
        break

with open(RUN_DIR/"history.json", "w") as f:
    json.dump(history, f)
print("CSV saved:", CSV_PATH)


CSV log  → runs/mtl_efficientnet_b0_20251114-171855/metrics.csv


Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

[Train] Epoch 1/100:   0%|          | 0/352 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# == POST-TRAIN: đọc CSV → vẽ & LƯU accuracy/loss vào images/ ==
import csv, numpy as np, matplotlib.pyplot as plt

epochs, train_accs, val_accs, train_losses, val_losses = [], [], [], [], []
with open(CSV_PATH, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs.append(int(row["epoch"]))
        train_accs.append(float(row["train_acc"]))
        val_accs.append(float(row["val_acc"]))
        train_losses.append(float(row["train_loss"]))
        val_losses.append(float(row["val_loss"]))

epochs = np.array(epochs)
train_accs = np.array(train_accs); val_accs = np.array(val_accs)
train_losses = np.array(train_losses); val_losses = np.array(val_losses)

# Accuracy
ACC_PNG = IMAGES_DIR / "accuracy_curve.png"
plt.figure(figsize=(7,4))
plt.plot(epochs, train_accs, marker='o', label="Train Acc")
plt.plot(epochs, val_accs, marker='o', label="Val Acc")
best_idx = int(np.argmax(val_accs))
plt.scatter([epochs[best_idx]], [val_accs[best_idx]], s=80, zorder=5,
            label=f"Best@{epochs[best_idx]}:{val_accs[best_idx]:.4f}")
plt.title("Accuracy vs Epoch"); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(ACC_PNG, dpi=150); plt.show()
print("Saved:", ACC_PNG)

# Loss
LOSS_PNG = IMAGES_DIR / "loss_curve.png"
plt.figure(figsize=(7,4))
plt.plot(epochs, train_losses, marker='o', label="Train Loss")
plt.plot(epochs, val_losses, marker='o', label="Val Loss")
plt.title("Loss vs Epoch"); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(LOSS_PNG, dpi=150); plt.show()
print("Saved:", LOSS_PNG)


In [ ]:
# Load BEST checkpoint để đánh giá
best_ckpt_path = CKPT_DIR / CKPT_BEST_NAME
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()

# Dự đoán test
all_preds, all_targets = [], []
with torch.no_grad(), torch.amp.autocast('cuda' if device.type=='cuda' else 'cpu'):
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        logits = model(imgs)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(labels.numpy())

all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

test_acc = float((all_preds == all_targets).mean())
print(f"🎯 Test Accuracy: {test_acc:.4f}")

# Lưu classification report
REPORT_TXT = RUN_DIR / "classification_report.txt"
with open(REPORT_TXT, "w", encoding="utf-8") as f:
    f.write(f"Test acc: {test_acc:.6f}\n\n")
    f.write(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, digits=4))
print("Saved:", REPORT_TXT)

# Confusion matrix với tuỳ chọn annotate theo tỷ lệ sai < ngưỡng
cm = confusion_matrix(all_targets, all_preds)
row_sums = cm.sum(axis=1, keepdims=True).astype(np.float64)
pct = (cm / np.clip(row_sums, 1, None)) * 100.0  # % theo hàng (true class)

CM_PNG = RUN_DIR / "confusion_matrix.png"
fig = plt.figure(figsize=(max(8, len(CLASS_NAMES)*0.35), max(6, len(CLASS_NAMES)*0.35)))
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(CLASS_NAMES))
plt.xticks(tick_marks, CLASS_NAMES, rotation=90)
plt.yticks(tick_marks, CLASS_NAMES)

# Annotate: chỉ in lên các ô SAI có % < ERR_ANNOTATE_UNDER_PCT
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    if i == j:
        # In số đúng nhỏ gọn (tuỳ thích)
        if cm[i, j] > 0:
            plt.text(j, i, f"{cm[i,j]}", ha="center", va="center", fontsize=7, color="black")
    else:
        p = pct[i, j]
        if p > 0 and p <= ERR_ANNOTATE_UNDER_PCT:  # tuỳ theo yêu cầu “dưới bao nhiêu % thì vẽ”
            plt.text(j, i, f"{cm[i,j]} ({p:.1f}%)", ha="center", va="center", fontsize=7, color="red")

plt.tight_layout()
plt.ylabel("True")
plt.xlabel("Predicted")
plt.savefig(CM_PNG, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", CM_PNG)


In [ ]:
#Biểu đồ cột Precision / Recall / F1 theo từng lớp
#👉 Ý nghĩa:
#Cột xanh – Precision (độ chính xác khi model dự đoán lớp đó)
#Cột cam – Recall (tỷ lệ phát hiện đúng lớp đó)
#Cột đỏ – F1-score (trung bình điều hòa của Precision và Recall).
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report

# lấy kết quả chi tiết thành dataframe
report_dict = classification_report(all_targets, all_preds, target_names=CLASS_NAMES, output_dict=True)
df = pd.DataFrame(report_dict).T.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")

plt.figure(figsize=(10, max(6, len(df)*0.3)))
df[["precision","recall","f1-score"]].plot(kind='bar', figsize=(10, max(6, len(df)*0.35)))
plt.title("Biểu đồ cột Precision / Recall / F1 theo từng lớp")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=45, ha='right')
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

REPORT_PNG = IMAGES_DIR / "classification_metrics_bar.png"
plt.savefig(REPORT_PNG, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", REPORT_PNG)


In [ ]:
# Biểu đồ đường F1-score sắp xếp theo thứ tự
f1_sorted = df["f1-score"].sort_values(ascending=False)
plt.figure(figsize=(10,5))
sns.lineplot(x=f1_sorted.index, y=f1_sorted.values, marker="o")
plt.title("FBiểu đồ đường F1-score sắp xếp theo thứ tự từng class")
plt.xticks(rotation=45, ha='right')
plt.ylabel("F1-score")
plt.ylim(0,1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
F1_PNG = IMAGES_DIR / "f1_sorted.png"
plt.savefig(F1_PNG, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", F1_PNG)


In [ ]:
#Thu thập logits dự đoán (bổ sung sau khi load model) chạy lại model 1 vòng trên tập test để lấy xác suất (logits)
import torch
from tqdm.auto import tqdm
import numpy as np
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# model đã load checkpoint "best" và đặt model.eval()
all_logits_list, all_targets_list = [], []

model.eval()
with torch.no_grad(), torch.amp.autocast('cuda' if device.type=='cuda' else 'cpu'):
    for imgs, labels in tqdm(test_loader, desc="Collecting logits"):
        imgs = imgs.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        logits = model(imgs)                    # logits chưa softmax
        all_logits_list.append(logits.cpu())
        all_targets_list.append(labels)

# Gộp tất cả lại
all_preds_logits = torch.cat(all_logits_list).numpy()   # shape: [N, NUM_CLASSES]
all_targets      = torch.cat(all_targets_list).numpy()  # shape: [N]

print("Collected logits:", all_preds_logits.shape)


In [ ]:
# dùng xác suất softmax từ logits ở trên để tính ROC/AUC.
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# Binarize ground truth theo số lớp
y_true_bin = label_binarize(all_targets, classes=list(range(NUM_CLASSES)))

# Softmax để lấy xác suất từng lớp
y_pred_prob = torch.softmax(torch.tensor(all_preds_logits), dim=1).numpy()

fpr = dict(); tpr = dict(); roc_auc = dict()

# Tính ROC-AUC cho từng lớp
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Macro-average ROC
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= NUM_CLASSES
roc_auc_macro = auc(all_fpr, mean_tpr)

# Vẽ ROC macro
plt.style.use("seaborn-v0_8-whitegrid")
plt.figure(figsize=(6,5), dpi=300)
plt.plot(all_fpr, mean_tpr, color='b', lw=2,
         label=f"Macro-average ROC (AUC = {roc_auc_macro:.3f})")
plt.plot([0,1], [0,1], 'k--', lw=1)
plt.title("ROC Curve (Macro-average)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

ROC_PNG = IMAGES_DIR / "roc_macro.png"
plt.savefig(ROC_PNG, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", ROC_PNG)


In [ ]:
# ==== ROC Top-K lớp kém nhất (AUC thấp) + lưu CSV AUC ====
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

K = 5  # số lớp kém nhất muốn hiển thị

# 1) Chuẩn bị xác suất dự đoán (từ logits) & nhãn one-vs-rest
y_true_bin = label_binarize(all_targets, classes=list(range(NUM_CLASSES)))   # shape [N, C]
y_pred_prob = torch.softmax(torch.tensor(all_preds_logits), dim=1).numpy()   # shape [N, C]

# 2) Tính ROC & AUC từng lớp
fpr, tpr, roc_auc = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# 3) Bảng AUC theo lớp, sắp xếp tăng dần để lấy lớp kém nhất
auc_df = pd.DataFrame({
    "class_idx": list(range(NUM_CLASSES)),
    "class_name": [CLASS_NAMES[i] for i in range(NUM_CLASSES)],
    "auc": [roc_auc[i] for i in range(NUM_CLASSES)]
}).sort_values("auc", ascending=True).reset_index(drop=True)

# Lưu CSV AUC để báo cáo
AUC_CSV = IMAGES_DIR / "roc_auc_per_class.csv"
auc_df.to_csv(AUC_CSV, index=False, encoding="utf-8-sig")
print("Saved AUC table →", AUC_CSV)

# 4) Vẽ ROC cho Top-K lớp kém nhất
worst = auc_df.head(K)
colors = plt.cm.tab10.colors  # bảng màu đẹp, phân biệt tốt

plt.figure(figsize=(6.5, 5.5), dpi=300)
for idx, (cls_idx, cls_name, auc_val) in enumerate(worst[["class_idx","class_name","auc"]].values):
    c = colors[idx % len(colors)]
    plt.plot(fpr[cls_idx], tpr[cls_idx], label=f"{cls_name} (AUC={auc_val:.3f})", lw=2, color=c)

# Macro-average (tham khảo)
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= NUM_CLASSES
roc_auc_macro = auc(all_fpr, mean_tpr)
plt.plot(all_fpr, mean_tpr, color="black", lw=2, linestyle="--",
         label=f"Macro-avg (AUC={roc_auc_macro:.3f})")

# Đường chéo ngẫu nhiên
plt.plot([0, 1], [0, 1], "k:", lw=1)

plt.title(f"ROC – Top {K} Worst Classes (lowest AUC)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
# Legend gọn: nếu quá dài, để outside phải
plt.legend(loc="lower right", fontsize=8, frameon=False)
plt.grid(alpha=0.3)
plt.tight_layout()

ROC_TOPK_PNG = IMAGES_DIR / f"roc_top{K}_worst.png"
plt.savefig(ROC_TOPK_PNG, dpi=300, bbox_inches="tight")
plt.show()
print("Saved plot →", ROC_TOPK_PNG)

# 5) In nhanh danh sách lớp kém để chèn vào báo cáo
display(worst)


In [ ]:
# ==== CONFUSION MATRIX — High-res, dễ nhìn, diagonal rõ, 2 dòng/ô ====
from sklearn.metrics import confusion_matrix
import numpy as np, matplotlib.pyplot as plt, itertools

cm = confusion_matrix(all_targets, all_preds)
labels = CLASS_NAMES
n = len(labels)

# % theo hàng (từng lớp thật)
row_sums = cm.sum(axis=1, keepdims=True).astype(np.float64)
pct = (cm / np.clip(row_sums, 1, None)) * 100.0  # % sai/đúng theo hàng

# Kích thước tỉ lệ số lớp để chữ không dính
fig_w = max(10, n * 0.42)
fig_h = max(8,  n * 0.42)

CM_PNG = IMAGES_DIR / "confusion_matrix.png"

fig = plt.figure(figsize=(fig_w, fig_h), dpi=600)
ax  = plt.gca()

# Màu nền sáng, đường chéo xanh đậm nổi bật
# 'YlGnBu' cho nền vàng nhạt → xanh lam đậm
im = ax.imshow(cm, interpolation='nearest', cmap='YlGnBu', vmin=0)
cbar = plt.colorbar(im, fraction=0.046, pad=0.02)
cbar.ax.tick_params(labelsize=8)

ax.set_title("Confusion Matrix", fontsize=14, pad=12)
ax.set_xticks(np.arange(n), labels=labels, fontsize=7, rotation=45, ha='right')
ax.set_yticks(np.arange(n), labels=labels, fontsize=7)
ax.set_xlabel("Predicted", fontsize=10)
ax.set_ylabel("True", fontsize=10)

# Lưới mảnh giúp tách ô
ax.set_xticks(np.arange(-.5, n, 1), minor=True)
ax.set_yticks(np.arange(-.5, n, 1), minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=0.6)
ax.tick_params(which="minor", bottom=False, left=False)

# Annotate từng ô:
# - Đường chéo (đúng): 2 dòng => số đúng + % đúng; dùng chữ trắng đậm cho nổi trên ô xanh đậm
# - Off-diagonal: chỉ annotate nếu % sai >= ANNOTATE_MIN_ERROR_PCT; chữ đỏ, 2 dòng => số sai + % sai (nguyên)
for i, j in itertools.product(range(n), range(n)):
    if i == j:
        if cm[i, j] > 0:
            p_correct = int(round(pct[i, j]))
            ax.text(j, i, f"{cm[i,j]}\n{p_correct}%",
                    ha="center", va="center",
                    fontsize=6.6, color="white", fontweight="bold")
    else:
        p_err = pct[i, j]
        if p_err >= ANNOTATE_MIN_ERROR_PCT and cm[i, j] > 0:
            ax.text(j, i, f"{cm[i,j]}\n{int(round(p_err))}%",
                    ha="center", va="center",
                    fontsize=6.4, color="#F30000")  # đỏ đậm cho lỗi

plt.tight_layout()
plt.savefig(CM_PNG, dpi=600, bbox_inches="tight")
plt.show()
print("Saved (600 DPI):", CM_PNG)


In [ ]:
#Top-K Accuracy (K=1,3,5)
def top_k_accuracy(logits, labels, k=3):
    topk = torch.topk(logits, k, dim=1).indices
    correct = (topk == labels.unsqueeze(1)).any(dim=1)
    return correct.float().mean().item()

model.eval(); all_logits = []
with torch.no_grad(), torch.amp.autocast('cuda'):
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        all_logits.append(logits.cpu())
all_logits = torch.cat(all_logits)
all_labels = torch.tensor(all_targets)

for k in [1,3,5]:
    acc = top_k_accuracy(all_logits, all_labels, k)
    print(f"Top-{k} Accuracy: {acc*100:.2f}%")


In [ ]:
import random, matplotlib.pyplot as plt, time

def show_random_predictions(n=12, save=True):
    idxs = random.sample(range(len(test_ds)), min(n, len(test_ds)))
    rows = math.ceil(n / 4)
    plt.figure(figsize=(12, 3*rows))
    model.eval()

    correct = 0
    for k, idx in enumerate(idxs, 1):
        img, label = test_ds[idx]
        with torch.no_grad(), torch.amp.autocast('cuda' if device.type=='cuda' else 'cpu'):
            logits = model(img.unsqueeze(0).to(device).to(memory_format=torch.channels_last))
            prob = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        pred = int(prob.argmax())
        conf = float(prob[pred])
        ok = (pred == label)
        correct += int(ok)

        # unnormalize
        std = torch.tensor(IMAGENET_STD).view(3,1,1)
        mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
        img_vis = (img * std + mean).clamp(0,1).permute(1,2,0).numpy()

        plt.subplot(rows, 4, k)
        plt.imshow(img_vis)
        title = f"P:{CLASS_NAMES[pred]} ({conf:.2f})\nT:{CLASS_NAMES[label]}"
        plt.title(title, color=("g" if ok else "r"), fontsize=9)
        plt.axis("off")

    acc_sample = correct / len(idxs)
    plt.suptitle(f"Random {len(idxs)} — Acc: {acc_sample:.3f}", y=1.02, fontsize=12)
    plt.tight_layout()

    if save:
        fname = IMAGES_DIR / f"random_predictions_{int(time.time())}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        print("Saved:", fname)
    plt.show()
    return acc_sample

# ví dụ:
show_random_predictions(n=50, save=True)
